# NTDS Gait → Metadata Probe  (Figure 2 reproduction)

Predict **age, BMI, and gender** from gait embeddings of the S3 `.ntds` corpus — the ASBV
panels of the paper's **Figure 2** (minus VAT, which we lack).

The notebook surfaces every stage so you can inspect the intermediate data as it is produced:
subset → **live forward pass into embeddings** → embedding table → cross-validated probe → plots.

**Two halves:**
- **Extraction (§1–3)** — a small *live* demo (a few clips) so you can watch the model run. The
  full ~20k-clip / 5-task extraction is a background script (`stream_extract`), **not** run here.
- **Analysis (§4–6) — start here** if you just want results: it loads the precomputed embedding
  parquet and runs the probe + plots; no extraction, no 3.4 h wait.

Reusable logic: `model/inference/ntds_embeddings.py` (streaming v5 extractor) and
`model/inference/ntds_probe.py` (nested-CV probe + plots). This notebook is the viewable interface.

> `.ntds` clips stream from S3 and are **deleted right after embedding** — nothing kept on disk.
> Needs a `.env` with AWS creds. Embeddings are **Variant-5, 1024-d** (the paper's pooling).

In [2]:
import os, sys
ROOT = os.path.abspath("..")
if ROOT not in sys.path: sys.path.insert(0, ROOT)
os.chdir(ROOT)                      # resolve data/ and results/ from the repo root
import numpy as np, pandas as pd, torch
from IPython.display import Image, display
from model.inference.ntds_embeddings import (
    load_config, select_subset, load_model, preprocess_ntds, embed_chunks,
    stream_extract, FIVE_TASKS)
from model.inference import ntds_probe

cfg = load_config()
model = load_model(cfg)
print(f"device={cfg.device}  bucket={cfg.bucket}")
print(f"manifest clips={len(cfg.manifest):,}  five tasks={FIVE_TASKS}")

device=mps  bucket=wis-p10k-378653386710-us-east-1-an
manifest clips=56,473  five tasks=['self_selected_gait_speed', 'tm_3kmh', 'stationary_walk', 'romberg_closed', 'sit_to_stand']


## 1 · Select subset (side-camera, 5 tasks) — for the live demo
Sampled by participant from the manifest: side cameras only (the model's training POV,
`is_front_facing==False`), the paper's **5 motor tasks**, labeled with age/gender/height/weight.
Here we take a tiny pool just to demo the forward pass; the full run uses `n_participants=None`.

In [ ]:
subset = select_subset(cfg, n_participants=20, activities=FIVE_TASKS)   # small pool for the live demo
print(f"{len(subset)} clips · {subset.participant_id.nunique()} participants (demo pool)")
display(subset.activity.value_counts().rename("clips"))
subset[["clip","participant_id","activity","age_at_session","gender","height_cm","weight_kg","bmi"]].head()

## 2 · Forward pass into embeddings (one clip, live)
Stream one clip from S3, run the DSTformer forward pass, inspect the tensors — then delete the
temp file. The embedding is **Variant-5** pooling of the encoder's `block_embeddings` → **1024-d**.

In [ ]:
import tempfile, boto3
row = subset.iloc[0]
tmp = os.path.join(tempfile.mkdtemp(), row["clip"])
b, k = row["ntds_uri"].replace("s3://", "").split("/", 1)
boto3.client("s3").download_file(b, k, tmp)

chunks = preprocess_ntds(cfg, tmp, max_chunks=1)
print("preprocessed input :", chunks.shape, "  (chunks, frames, joints, channels)")
with torch.no_grad():
    recon, bundle = model(torch.tensor(chunks[:1], dtype=torch.float32).to(cfg.device))
print("reconstructed XYZ  :", tuple(recon.shape))
print("block_embeddings   :", tuple(bundle["block_embeddings"].shape), " (frames, joints, 128 features)")
emb = embed_chunks(cfg, model, chunks)      # Variant-5 pool of block_embeddings -> 1024-d
print("v5 embedding       :", emb.shape, " first 6:", emb[:6].round(3))
os.remove(tmp); print("temp file deleted  :", not os.path.exists(tmp))

## 3 · Stream a handful (see the loop)
`stream_extract` prefetch-downloads → embeds (v5) → **deletes** each clip, appending to a resumable
parquet. Small live slice here; the full ~20k-clip run is the background script.

In [3]:
demo = stream_extract(cfg, subset.head(6), model,
                      out_parquet="results/_ntds_live_demo.parquet", max_chunks=1,
                      progress=lambda i, n, msg: print(f"  [{i}/{n}] {msg}"))
print("streamed", len(demo), "clips — no .ntds kept on disk")
demo[["clip","activity","age","gender_male","bmi","emb_0","emb_1","emb_2"]].head(6)

NameError: name 'subset' is not defined

## 4 · Analysis — **start here** (loads embeddings; no extraction needed)
Loads the precomputed embedding parquet. Prefers the full 5-task run
(`results/ntds_5task_full_emb.parquet`, written by the background script); falls back to the
700-participant mid run if the full one isn't ready yet. One **1024-d** v5 embedding per clip + labels.

In [7]:
FULL = "results/ntds_5task_full_emb.parquet"
MID  = "results/ntds_5task_mid_emb.parquet"
emb_path = FULL if os.path.exists(FULL) else MID
PREFIX   = "ntds_5task_full" if emb_path == FULL else "ntds_5task_mid"
emb_df   = pd.read_parquet(emb_path)
print("loaded", emb_path, "·", emb_df.shape, "·", emb_df.participant_id.nunique(), "participants")
print("age", f"{emb_df.age.min():.0f}-{emb_df.age.max():.0f}",
      "· gender(male=1)", emb_df.gender_male.value_counts().to_dict(),
      "· bmi", f"{emb_df.bmi.min():.1f}-{emb_df.bmi.max():.1f}",
      "· activities", sorted(emb_df.activity.unique()))
emb_df.iloc[:5, :8]

loaded results/ntds_5task_full_emb.parquet · (19118, 1031) · 3643 participants
age 20-79 · gender(male=1) {0.0: 9782, 1.0: 9336} · bmi 16.1-49.6 · activities ['romberg_closed', 'self_selected_gait_speed', 'sit_to_stand', 'stationary_walk', 'tm_3kmh']


,clip,participant_id,test_id,activity,age,gender_male,bmi,emb_0
0,K4A000234514612_HPECONFIG_2025-12-25T09_32_10Z...,3454bfaa-7272-40a9-8d62-073b2f09beb7,f370ef56-67ac-4b1b-9a7b-e84adf2777a8,romberg_closed,57.0,1.0,26.703624,0.018288
1,K4A000234514612_HPECONFIG_2025-12-25T09_19_53Z...,39584500-0ed1-47c6-9ad9-719a7f9d1469,f8b39381-b95f-49f5-adb2-0c189db7406d,sit_to_stand,46.0,1.0,27.265306,-0.014049
2,K4A000234514612_HPECONFIG_2025-12-25T09_30_45Z...,3454bfaa-7272-40a9-8d62-073b2f09beb7,f370ef56-67ac-4b1b-9a7b-e84adf2777a8,sit_to_stand,57.0,1.0,26.703624,-0.034758
3,K4A000234514612_HPECONFIG_2025-12-25T09_21_17Z...,39584500-0ed1-47c6-9ad9-719a7f9d1469,f8b39381-b95f-49f5-adb2-0c189db7406d,romberg_closed,46.0,1.0,27.265306,0.050184
4,K4A000234514612_HPECONFIG_2025-12-25T09_29_24Z...,3454bfaa-7272-40a9-8d62-073b2f09beb7,f370ef56-67ac-4b1b-9a7b-e84adf2777a8,stationary_walk,57.0,1.0,26.703624,-0.027865


## 5 · Probe: age & BMI (sex-stratified) + gender — nested CV (paper protocol)
Nested subject-level CV (outer 5-fold, inner 4-fold α-tuning), late-fusion across the 5 tasks.
Age & BMI are trained **within each sex**; gender is un-stratified. `n_seeds=5` here for a quick
interactive read; the background script uses **15 seeds** for the paper-grade CIs saved to `results/`.
`nested_report` returns the metrics table *and* writes the plots.

In [8]:
metrics, plot_paths = ntds_probe.nested_report(emb_df, out_dir="results", prefix=PREFIX, n_seeds=5)
display(metrics)

    [gender_male/clas] model picks: {'lin': 25}
    [age/reg] model picks: {'lin': 25}
    [age/reg] model picks: {'lin': 25}
    [bmi/reg] model picks: {'lin': 25}
    [bmi/reg] model picks: {'lin': 25}


,target,sex,metric,mean,std
0,Gender,all,auc,0.998,0.000
1,Age,male,pearson,0.687,0.003
2,Age,female,pearson,0.672,0.001
3,BMI,male,pearson,0.873,0.001
4,BMI,female,pearson,0.900,0.001


## 6 · Result plots → `results/`
Sex-stratified predicted-vs-true scatter (age, BMI; ♂ blue / ♀ orange) and the gender ROC.

In [6]:
for name in ["age", "bmi", "gender"]:
    display(Image(filename=plot_paths[name]))

FileNotFoundError: [Errno 2] No such file or directory: 'results/ntds_5task_full_age_scatter.png'